# 05 — LoRA adaptation for future transaction value

This is the primary adaptation experiment. The label is `log1p(sum(abs(amount)))` across the next 180 days after each cutoff. It measures future transaction volume, not lifetime value in the business sense: the source contains neither revenue nor retention, and the label is only a defensible proxy for future account activity.

The frozen embedding’s initial held-out MAE was slightly lower than the tabular Ridge baseline, so this is the most defensible task on which to test whether parameter-efficient adaptation adds signal.

## Optimisation protocol

The backbone remains frozen and the same History Encoder-only LoRA scope is used as in notebook 04. The new one-dimensional task head has learning rate `1e-3`; LoRA parameters use `3e-4`. The loss is Smooth L1 on the already log-transformed target. Gradient norm is clipped at 1.0. Rank and checkpoint are selected by validation MAE with patience eight; the held-out test is evaluated only after selection.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/FinancialBertForTransactions')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_ROOT = Path('/content/drive/MyDrive/FinancialBertForTransactions')
else:
    PERSIST_ROOT = PROJECT_ROOT
CHECKPOINT = PERSIST_ROOT / 'checkpoints' / 'pragma_lite_mlm' / 'best.pt'
REPORT_DIR = PERSIST_ROOT / 'reports'
TASK_TABLE = REPORT_DIR / 'future_value_task_table.parquet'
BASELINE_REPORT = REPORT_DIR / 'future_value_tabular_baseline.json'
FROZEN_REPORT = REPORT_DIR / 'future_value_frozen_probe.json'
ADAPTER_DIR = PERSIST_ROOT / 'adapters' / 'future_value'
assert all(path.exists() for path in (CHECKPOINT, TASK_TABLE, BASELINE_REPORT, FROZEN_REPORT)), 'Run notebook 03 first.'

In [ ]:
subprocess.run([
    sys.executable, 'scripts/run_lora_finetune.py',
    '--task', 'future_value',
    '--checkpoint', str(CHECKPOINT),
    '--task-table', str(TASK_TABLE),
    '--output-dir', str(ADAPTER_DIR),
    '--baseline-report', str(BASELINE_REPORT),
    '--frozen-probe-report', str(FROZEN_REPORT),
    '--ranks', '4,8,16', '--max-epochs', '50', '--patience', '8',
    '--device', 'cuda' if IN_COLAB else 'cpu',
], cwd=PROJECT_ROOT, check=True)

The saved adapter contains no copy of frozen backbone weights. It records the base checkpoint SHA-256, task-table path, LoRA configuration, trainable parameter count, task-head weights, validation score, and training configuration. This makes it portable but explicitly tied to the exact base model and target definition used here.